## Session affinity, multi-port & Service vs Ingress

### Session affinity

By default every connection is load-balanced independently — back-to-back requests may hit different pods. For sticky sessions (pod-local session state, shopping carts):

```yaml
spec:
  sessionAffinity: ClientIP
  sessionAffinityConfig:
    clientIP: { timeoutSeconds: 10800 }   # default 3h
```

`ClientIP` hashes by source IP and pins that client to one pod. It's the **only** affinity mode Kubernetes offers — cookie/JWT stickiness needs an Ingress or mesh. Warning: many clients behind one NAT all look like one IP, so affinity stops helping — use it for *internal* traffic where callers have distinct IPs.

### Multi-port services

One Service can expose several ports; each entry becomes its own rule, and **each must be named**:

```yaml
ports:
  - { name: http, port: 80, targetPort: 8080 }
  - { name: grpc, port: 9090, targetPort: 9090 }
```

### Service vs Ingress

A Service works at **L4** — TCP/UDP ports and a selector. It knows nothing of hostnames, paths, TLS, or HTTP. For HTTP routing — "`/api/*` here, `/static/*` there, TLS on the front, one public IP" — you want an **Ingress** (or the **Gateway API**): a Kubernetes object an **ingress controller** (nginx, Traefik, Istio) reads to configure an HTTP proxy in front of your Services.

| Layer | Object | Speaks |
|---|---|---|
| L4 in-cluster | Service (ClusterIP) | TCP/UDP, selector |
| L4 edge | Service (NodePort/LB) | TCP/UDP at the edge |
| L7 HTTP | Ingress + controller | host/path routing, TLS |

**Service is the foundational primitive** — Ingress and mesh *use* Services to find pods, then add smarter routing. Notebook 09 covers Ingress properly. On our map, this section spans the **Networking** box from **Service** to **Ingress** — the handoff from L4 to L7.